<h1 style="color: green;">Home task : sentiment analysis</h1>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
from datasets import Dataset


2025-08-03 15:58:00.663663: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
path = "/Users/user/Desktop/Camp2025/lesson_21/data/rt-polarity.neg"

with open(path, "r", encoding='utf-8', errors='ignore') as f:
    content = f.read()
neg_text = content.splitlines()
print('Len of negative text = {:,}'.format (len(neg_text)))
for i in neg_text[:5]:
    print ('\n', i)

Len of negative text = 5,331

 simplistic , silly and tedious . 

 it's so laddish and juvenile , only teenage boys could possibly find it funny . 

 exploitative and largely devoid of the depth or sophistication that would make watching such a graphic treatment of the crimes bearable . 

 [garbus] discards the potential for pathological study , exhuming instead , the skewed melodrama of the circumstantial situation . 

 a visually flashy but narratively opaque and emotionally vapid exercise in style and mystification . 


In [3]:
path = "/Users/user/Desktop/Camp2025/lesson_21/data/rt-polarity.pos"

with open(path, "r", encoding='utf-8', errors='ignore') as f:
    content = f.read()
pos_text = content.splitlines()
print('Len of positive text = {:,}'.format (len(pos_text)))
for i in pos_text[:5]:
    print ('\n', i)

Len of positive text = 5,331

 the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal . 

 the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson's expanded vision of j . r . r . tolkien's middle-earth . 

 effective but too-tepid biopic

 if you sometimes like to go to the movies to have fun , wasabi is a good place to start . 

 emerges as something rare , an issue movie that's so honest and keenly observed that it doesn't feel like one . 


In [4]:
texts = neg_text + pos_text
labels = [0] * len(neg_text) + [1] * len(pos_text) 

X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)

In [5]:
vectorizer = CountVectorizer(min_df=5, max_features=50000, ngram_range=(1,2)).fit(texts)

X_train_vectorized = vectorizer.transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

In [6]:
def evaluate_model(y_true, y_pred):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-score': f1_score(y_true, y_pred)
    }

results = {}

## Logistic Regression

In [7]:
clf = LogisticRegression(max_iter=3000).fit(X_train_vectorized, y_train) 
y_pred_lr = clf.predict(X_test_vectorized)
results['Logistic Regression'] = evaluate_model(y_test, y_pred_lr)

## DistilBERT

In [8]:

sample_size = 2000
indices = np.random.choice(len(X_train), sample_size, replace=False)
X_train_sample = [X_train[i] for i in indices]
y_train_sample = [y_train[i] for i in indices]

train_dataset = Dataset.from_dict({
    "text": X_train_sample,
    "label": y_train_sample
})
test_dataset = Dataset.from_dict({
    "text": X_test,
    "label": y_test
})

In [9]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2133 [00:00<?, ? examples/s]

In [10]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    use_cpu=True 
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

trainer.train()

Step,Training Loss
10,0.703900
20,0.689200
30,0.691300
40,0.660100
50,0.668900
60,0.588400
70,0.506400
80,0.530100
90,0.459300
100,0.412400


TrainOutput(global_step=375, training_loss=0.3272378667195638, metrics={'train_runtime': 924.198, 'train_samples_per_second': 6.492, 'train_steps_per_second': 0.406, 'total_flos': 198701097984000.0, 'train_loss': 0.3272378667195638, 'epoch': 3.0})

In [11]:
predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=-1)


results['DistilBERT'] = {
    'Accuracy': accuracy_score(y_test, preds),
    'Precision': precision_score(y_test, preds),
    'Recall': recall_score(y_test, preds),
    'F1-score': f1_score(y_test, preds)
}

## Zero-Shot

In [12]:
zero_shot_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device="cpu")

sample_size = 2000
X_test_sample = pd.Series(X_test).sample(n=sample_size, random_state=42).reset_index(drop=True)
y_test_sample = pd.Series(y_test).iloc[X_test_sample.index].reset_index(drop=True)

candidate_labels = ["negative", "positive"]
label_to_int = {"negative": 0, "positive": 1}
predictions = []

for review in X_test_sample:
    result = zero_shot_classifier(review[:512], candidate_labels)
    predicted_label = label_to_int[result['labels'][0]]
    predictions.append(predicted_label)

results["Zero-Shot"] = {
    'Accuracy': accuracy_score(y_test_sample, predictions),
    'Precision': precision_score(y_test_sample, predictions),
    'Recall': recall_score(y_test_sample, predictions),
    'F1-score': f1_score(y_test_sample, predictions)
}

Device set to use cpu


In [13]:
df_results = pd.DataFrame(results).T 
df_results

,Accuracy,Precision,Recall,F1-score
Logistic Regression,0.769808,0.774102,0.764706,0.769375
DistilBERT,0.827942,0.864389,0.779645,0.819833
Zero-Shot,0.509000,0.510107,0.429429,0.466304


## Conclusions

In the course of this task, three approaches to text sentiment analysis were implemented: Logistic Regression, Zero-Shot Learning, and DistilBERT.

* Logistic Regression The main advantage is its interpretability and speed. However, the model does not take into account the context, which is a critical place for natural language processing.

* Zero-Shot Its strength is in its flexibility and ability to work with any classes formulated in the form of a hypothesis. However, it is usually slower and less accurate than specially trained models.

* DistilBERT is a simplified version of BERT that provides a balance between accuracy and speed. It uses the contextual representation of words and showed the highest quality of results among the three methods. The main disadvantage is the need for more cleaning resources compared to classical methods.

### The best results were achieved using the DistilBERT model, which best copes with understanding the context of the text, which is critically important for sentiment classification. At the same time, Logistic Regression can be recommended for quick tests or limited computing resources, and Zero-Shot - in cases where it is not possible to have a specially prepared data set. 